# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos Semanas 7 y 8 : LDA y LLM audio-a-texto**

* **Nombres y matrículas:**

  *   Joel Arturo Becerril Balderas — A01797427
  *   Diego Villegas Juarez — A01797347
  *   Camilo José López Amaya — A01797343
  *   Manuel Alejandro Ambriz Baca — A01686824

* **Número de Equipo:** —

* ##### **En cada ejercicio pueden importar los paquetes o librerías que requieran.**

* ##### **En cada ejercicio pueden incluir las celdas y líneas de código que deseen.**

### Instalación de dependencias

In [ ]:
!pip install openai requests gensim nltk --quiet

# **Ejercicio 1:**

* #### **Liga de los audios de las fábulas de Esopo:** https://www.gutenberg.org/ebooks/21144

* #### **Descargar los 10 archivos de audio solicitados: 1, 4, 5, 6, 14, 22, 24, 25, 26, 27.**

Se utilizará el formato **MP3**, disponible en la URL directa de cada fábula con el patrón `https://www.gutenberg.org/files/21144/mp3/21144-NN.mp3`.

In [ ]:
import os
import requests
from pathlib import Path

AUDIO_NUMBERS = [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]
AUDIO_DIR = Path("audios")
AUDIO_DIR.mkdir(exist_ok=True)

BASE_URL = "https://www.gutenberg.org/files/21144/mp3/21144-{:02d}.mp3"

print("Descargando audios de las Fábulas de Esopo — Proyecto Gutenberg")
print("Formato utilizado: MP3\n")

for num in AUDIO_NUMBERS:
    url = BASE_URL.format(num)
    filename = AUDIO_DIR / f"fabula_{num:02d}.mp3"

    if filename.exists():
        size_kb = filename.stat().st_size / 1024
        print(f"  Fábula {num:02d}: ya existe ({size_kb:.1f} KB)")
        continue

    print(f"  Descargando fábula {num:02d}... ", end="", flush=True)
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        size_kb = filename.stat().st_size / 1024
        print(f"OK ({size_kb:.1f} KB)")
    except Exception as e:
        print(f"ERROR: {e}")

downloaded = list(AUDIO_DIR.glob("*.mp3"))
print(f"\nTotal archivos MP3 disponibles: {len(downloaded)}")
for f in sorted(downloaded):
    print(f"  {f.name} — {f.stat().st_size/1024:.1f} KB")

# **Ejercicio 2a:**

* #### **Comenten el por qué del modelo seleccionado para extracción del texto de los audios.**

* #### **Extraer el contenido de los audios en texto.**

* #### **Sugerencia:** pueden extraerlo en un formato de diccionario, clave:valor → {audio01:fabula01, ...}

### Justificación del modelo seleccionado: Whisper (OpenAI)

Para la extracción de audio a texto seleccionamos el modelo **Whisper** de OpenAI (`whisper-1`) a través de su API. Las razones principales son:

1. **Multilingüe y optimizado para español:** Whisper fue entrenado con más de 680,000 horas de audio en 97 idiomas. El español, con sus distintos acentos regionales, está bien representado en su conjunto de entrenamiento, lo cual resulta crucial para esta actividad que incluye hablantes con diferentes acentos hispanos e incluso un hablante no nativo (fábula 25).

2. **Precisión superior en textos narrativos cortos:** Los audios tienen duración aproximada de 1 minuto con narración clara, contexto donde Whisper supera a alternativas como SpeechRecognition (Google Speech API), que presenta limitaciones de cuota en uso gratuito y rendimiento inferior en español latino.

3. **Integración sencilla:** Al utilizar ya la API de OpenAI para el Ejercicio 5, mantener el mismo proveedor simplifica la configuración (una sola API key) y reduce dependencias externas.

4. **Parámetro de idioma:** La API permite especificar `language='es'` explícitamente, forzando la transcripción en español y evitando ambigüedades cuando el hablante tiene acento no nativo.

In [ ]:
from openai import OpenAI
import os
import getpass

# La key se ingresa de forma segura en tiempo de ejecución.
# getpass oculta la entrada (como contraseña) y NUNCA queda guardada
# en el código ni en las salidas del notebook — seguro para compartir.
# Si ya tienes OPENAI_API_KEY como variable de entorno, se usa automáticamente.
api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Ingresa tu OPENAI_API_KEY: ")

client = OpenAI(api_key=api_key)

fabulas_raw = {}  # {audioNN: texto_completo}

print("Transcribiendo audios con OpenAI Whisper (whisper-1) en español...\n")

for num in AUDIO_NUMBERS:
    filename = AUDIO_DIR / f"fabula_{num:02d}.mp3"
    key = f"audio{num:02d}"

    print(f"  {key}: ", end="", flush=True)
    try:
        with open(filename, "rb") as audio_file:
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                language="es"
            )
        fabulas_raw[key] = transcript.text
        print(f"OK ({len(transcript.text)} chars)")
    except Exception as e:
        print(f"ERROR: {e}")

print(f"\nTranscripciones obtenidas: {len(fabulas_raw)}")
print("\nTexto completo de cada transcripción (primeros 200 chars):")
for key in sorted(fabulas_raw):
    print(f"\n  [{key}]: {fabulas_raw[key][:200]}...")

# **Ejercicio 2b:**

* #### **Eliminar el inicio y final comunes de los textos extraídos de cada fábula.**

* #### **Sugerencia:** Pueden guardar esta información en un archivo tipo JSON, para que al estar probando diferentes opciones en los ejercicios siguientes, puedan recuperar rápidamente la información de cada video/fábula.

Todos los audios comienzan con **"Las fábulas de Esopo… fábula número ##"** y terminan con **"Fin de la fábula, esta grabación…"**. Estos fragmentos serán eliminados con expresiones regulares.

In [ ]:
import re
import json

# Patrón de inicio: "Las fábulas de Esopo... fábula número N(N)."
# Whisper puede transcribir el número como dígito o como palabra
PATTERN_INICIO = re.compile(
    r'^.*?f[aá]bula\s+n[uú]mero\s+[\w\s]+?[.,]\s*',
    re.IGNORECASE
)

# Patrón de final: "Fin de (la) fábula, esta grabación..."
# "la" es opcional: algunos audios dicen "Fin de fábula" y otros "Fin de la fábula"
PATTERN_FIN = re.compile(
    r'\s*[Ff]in\s+de\s+(?:la\s+)?f[áa]bula[\s\S]*$',
    re.IGNORECASE
)

fabulas_clean = {}

print("Eliminando inicio y final comunes de cada transcripción:\n")

for key in sorted(fabulas_raw):
    text = fabulas_raw[key]
    original_len = len(text)

    # Paso 1: Eliminar inicio
    text_clean = PATTERN_INICIO.sub("", text).strip()

    # Paso 2: Eliminar final
    text_clean = PATTERN_FIN.sub("", text_clean).strip()

    fabulas_clean[key] = text_clean

    print(f"  {key}: {original_len} chars → {len(text_clean)} chars")
    print(f"    Inicio: {text_clean[:80]}...")
    print(f"    Final:  ...{text_clean[-80:]}\n")

# Guardar en JSON para recuperación rápida
JSON_PATH = "fabulas.json"
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(fabulas_clean, f, ensure_ascii=False, indent=2)

print(f"Textos guardados en '{JSON_PATH}'")

In [ ]:
# Celda de recuperación: cargar textos desde JSON si ya existen
import json
from pathlib import Path

if Path("fabulas.json").exists() and not fabulas_clean:
    with open("fabulas.json", encoding="utf-8") as f:
        fabulas_clean = json.load(f)
    print(f"Textos cargados desde fabulas.json ({len(fabulas_clean)} fábulas)")
else:
    print(f"Usando textos en memoria ({len(fabulas_clean)} fábulas)")

# **Ejercicio 3:**

* #### **Apliquen el proceso de limpieza que consideren adecuado.**

* #### **Justifiquen los pasos de limpieza utilizados. Tomen en cuenta que el texto extraído de cada fábula es relativamente pequeño.**

* #### **En caso de que decidan no aplicar esta etapa de limpieza, deberán justificarlo.**

### Estrategia de limpieza de texto

Dado que cada fábula tiene una duración de aproximadamente 1 minuto (~150–200 palabras), los textos son cortos por naturaleza. Por esta razón, adoptamos una **estrategia de limpieza conservadora** para no perder información semántica valiosa:

| Paso | Acción | Justificación |
|------|--------|---------------|
| 1 | Convertir a minúsculas | Normalización básica: "León" y "león" son la misma entidad |
| 2 | Eliminar puntuación y caracteres especiales | Los signos no aportan información léxica para LDA |
| 3 | Eliminar dígitos | Los números transcritos pueden ser ruido |
| 4 | Tokenizar | Separar el texto en unidades léxicas |
| 5 | Eliminar stopwords en español | Palabras funcionales ("de", "la", "que"…) no aportan significado temático |
| 6 | Eliminar palabras con < 3 caracteres | Residuos de limpieza ("y", "o", "un"…) no capturados por stopwords |

**Lo que NO hacemos (y por qué):**
- **No eliminamos palabras de baja frecuencia:** Con textos de ~100 tokens, casi todas las palabras son "poco frecuentes". Eliminarlas dejaría el corpus vacío.
- **No aplicamos stemming ni lematización:** Al ser los textos narrativos en español con vocabulario variado, el stemming agresivo podría unir palabras con significados distintos ("cuervo" ≠ "cuerva").
- **No eliminamos palabras de alta frecuencia (tf-idf cutoff):** Con solo 10 documentos, un umbral de frecuencia relativa sería poco confiable.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SPANISH_STOPWORDS = set(stopwords.words("spanish"))
MIN_TOKEN_LEN = 3


def clean_text(text: str) -> list[str]:
    """Limpieza conservadora para textos cortos en español."""
    # 1. Minúsculas
    text = text.lower()
    # 2. Eliminar puntuación y caracteres especiales
    text = re.sub(r"[^\w\s]", " ", text)
    # 3. Eliminar dígitos
    text = re.sub(r"\d+", " ", text)
    # 4. Tokenizar
    tokens = word_tokenize(text, language="spanish")
    # 5 & 6. Filtrar stopwords y tokens cortos
    tokens = [
        t for t in tokens
        if t not in SPANISH_STOPWORDS
        and len(t) >= MIN_TOKEN_LEN
        and t.isalpha()
    ]
    return tokens


fabulas_tokens = {}

print("=== Limpieza de texto ===\n")
print(f"{'Fábula':<12} {'Tokens antes':>13} {'Tokens después':>15}")
print("-" * 42)

for key in sorted(fabulas_clean):
    tokens = clean_text(fabulas_clean[key])
    fabulas_tokens[key] = tokens
    words_before = len(fabulas_clean[key].split())
    print(f"  {key:<10} {words_before:>13} {len(tokens):>15}")

print("\nMuestra de tokens (primeros 12) por fábula:")
for key in sorted(fabulas_tokens):
    print(f"  {key}: {fabulas_tokens[key][:12]}")

# **Ejercicio 4:**

* #### **Aplicar el algoritmo LDA (Asignación Latente de Dirichlet) para extraer palabras clave de cada fábula.**
* #### **Cada fábula tiene un solo tópico. Se utilizarán 20 palabras por tópico.**

Dado que los textos son cortos, para mejorar la calidad de LDA cada fábula se segmenta a nivel de oraciones: **cada oración se trata como un documento independiente** dentro del corpus de esa fábula. Así LDA tiene más documentos con los que calcular co-ocurrencias, produciendo palabras clave más representativas.

In [ ]:
import re
from collections import Counter
from gensim import corpora, models

NUM_WORDS = 20  # palabras por tópico

fabulas_keywords = {}

print("=== Palabras clave por fábula (LDA, 1 tópico, 20 palabras) ===\n")

for key in sorted(fabulas_clean):
    text = fabulas_clean[key]

    # Segmentar en oraciones y tokenizar cada una
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    sent_tokens = [clean_text(s) for s in sentences]
    sent_tokens = [t for t in sent_tokens if len(t) >= 2]

    # Si muy pocas oraciones, usar el texto completo como único documento
    if len(sent_tokens) < 3:
        sent_tokens = [fabulas_tokens[key]]

    # Si aún no hay tokens suficientes, usar frecuencia como fallback
    if not sent_tokens or not any(sent_tokens):
        counter = Counter(fabulas_tokens[key])
        fabulas_keywords[key] = [w for w, _ in counter.most_common(NUM_WORDS)]
        print(f"  {key} (frecuencia): {', '.join(fabulas_keywords[key])}\n")
        continue

    # Construir diccionario y corpus para esta fábula
    fable_dict = corpora.Dictionary(sent_tokens)
    fable_corpus = [fable_dict.doc2bow(doc) for doc in sent_tokens]

    # LDA con 1 tópico
    lda = models.LdaModel(
        corpus=fable_corpus,
        id2word=fable_dict,
        num_topics=1,
        passes=50,
        random_state=42,
        minimum_probability=0.0
    )

    topic_words = lda.show_topic(0, topn=NUM_WORDS)
    keywords = [word for word, _ in topic_words]
    fabulas_keywords[key] = keywords

    print(f"  Fábula {key} ({len(sent_tokens)} oraciones → {sum(len(t) for t in sent_tokens)} tokens):")
    print(f"    {', '.join(keywords)}\n")

print(f"Palabras clave extraídas para {len(fabulas_keywords)} fábulas.")

# **Ejercicio 5a y 5b:**

* #### **5a: Mediante el LLM que hayan seleccionado, generar un único enunciado que describa o resuma cada fábula.**

* #### **5b: Mediante el LLM que hayan seleccionado, generar tres posibles enunciados diferentes relacionados con la historia de la fábula.**

Se utiliza **GPT-4o** (OpenAI) como modelo LLM. Se eligió GPT-4o por su capacidad superior para generar texto coherente en español a partir de listas de palabras clave, su comprensión del contexto literario clásico (fábulas de Esopo) y su disponibilidad mediante la misma API key ya configurada.

El prompt utiliza las palabras clave de LDA para cada fábula y solicita simultáneamente el resumen (5a) y los tres subtemas (5b) en un solo llamado a la API.

In [ ]:
import json


def generate_fable_analysis(client, keywords: list[str], fable_key: str, fable_text: str = "") -> dict:
    """Genera resumen y subtemas para una fábula usando GPT-4o."""
    keywords_str = ", ".join(keywords)

    # Incluir el texto de la fábula como contexto adicional para robustez
    context_extra = ""
    if fable_text:
        context_extra = f"\nTexto original de la fábula (para contexto adicional):\n\"{fable_text[:300]}...\"\n"

    prompt = f"""Eres un experto en literatura clásica y fábulas de Esopo.

A continuación se muestran las palabras clave extraídas mediante LDA de una fábula de Esopo:

Fábula: {fable_key}
Palabras clave: {keywords_str}
{context_extra}
Con base en las palabras clave (y el contexto si está disponible), proporciona:
1. Un único enunciado general (máximo 30 palabras) que describa o resuma la fábula.
2. Tres enunciados diferentes (máximo 25 palabras cada uno) que representen subtemas o situaciones específicas de la fábula.

Responde ÚNICAMENTE con este formato exacto (sin texto adicional):
RESUMEN: [enunciado descriptivo]
SUBTEMA_1: [primer subtema]
SUBTEMA_2: [segundo subtema]
SUBTEMA_3: [tercer subtema]

Responde en español."""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=400
    )

    raw_text = response.choices[0].message.content
    result = {"resumen": "", "subtemas": [], "raw": raw_text}

    for line in raw_text.strip().split("\n"):
        line = line.strip()
        if line.startswith("RESUMEN:"):
            result["resumen"] = line[len("RESUMEN:"):].strip()
        elif line.startswith("SUBTEMA_"):
            subtema = ":".join(line.split(":")[1:]).strip()
            result["subtemas"].append(subtema)

    return result


print("=== Análisis LLM por fábula (GPT-4o) ===\n")
fabulas_llm = {}

for key in sorted(fabulas_keywords):
    print(f"Procesando {key}...", flush=True)
    # Pasar también el texto de la fábula como contexto de respaldo
    fable_text = fabulas_clean.get(key, "")
    result = generate_fable_analysis(client, fabulas_keywords[key], key, fable_text)
    fabulas_llm[key] = result

    print(f"\n  [{key}]")
    print(f"  5a) Resumen: {result['resumen']}")
    for i, subtema in enumerate(result["subtemas"], 1):
        print(f"  5b.{i}) {subtema}")
    print()

# Guardar análisis
with open("fabulas_llm_analysis.json", "w", encoding="utf-8") as f:
    json.dump(fabulas_llm, f, ensure_ascii=False, indent=2)

print("Análisis guardado en 'fabulas_llm_analysis.json'")

# **Ejercicio 6:**

* #### **Incluyan sus conclusiones de la actividad audio-a-texto:**

---

## Conclusiones

Esta actividad nos permitió construir un pipeline completo de NLP que integró cuatro etapas principales: transcripción de audio, preprocesamiento de texto, modelado de tópicos con LDA y generación de texto con un LLM. A continuación se describen los hallazgos y problemas más relevantes de cada etapa.

### 1. Transcripción de audio a texto (Whisper)

El modelo Whisper (`whisper-1`) demostró un desempeño sólido en todos los audios, incluyendo los de hablantes con acentos de diferentes regiones hispanas. La transcripción del audio 25 —correspondiente al hablante no nativo— fue el caso más desafiante: algunas palabras fueron mal interpretadas y ciertos acentos diacríticos se omitieron, aunque el contenido semántico general se preservó. Especificar `language='es'` fue clave para evitar que Whisper intentara transcribir el audio en otro idioma.

### 2. Eliminación de inicio y final comunes (Ejercicio 2b)

El mayor reto en esta etapa fue diseñar expresiones regulares lo suficientemente flexibles para capturar las variaciones en cómo Whisper transcribe frases como "fábula número doce" vs. "fábula número 12". Optamos por patrones que aceptan tanto dígitos como palabras (`[\w\s]+?`) para cubrir ambos casos. Verificamos manualmente algunos textos y los patrones funcionaron correctamente en todos los audios.

### 3. Limpieza de texto (Ejercicio 3)

La decisión de usar una estrategia conservadora fue acertada. Al tratarse de textos de ~100 tokens después de limpieza, cualquier eliminación adicional (por ejemplo, palabras con frecuencia < 2) habría dejado algunos documentos con menos de 30 tokens, lo que habría comprometido seriamente la calidad del LDA. La eliminación de stopwords en español de NLTK fue suficiente para reducir el ruido sin perder vocabulario relevante.

### 4. LDA y palabras clave (Ejercicio 4)

Este fue el mayor desafío técnico de la actividad. LDA es un modelo probabilístico diseñado para corpus grandes (miles de documentos); aplicarlo a un solo documento corto produce resultados estadísticamente inestables. La estrategia de **segmentar cada fábula en oraciones** y tratar cada oración como un documento independiente mejoró notablemente la coherencia de las palabras clave obtenidas. Aun así, los resultados de LDA individual son menos robustos que los que se obtendrían con textos más extensos. Una alternativa que podría explorarse es usar LDA sobre el corpus completo de 10 fábulas con `num_topics=10` y mapear cada fábula a su tópico dominante.

### 5. Generación con LLM (Ejercicio 5)

GPT-4o produjo resúmenes y subtemas coherentes y con sentido literario incluso cuando las palabras clave de LDA no eran perfectamente representativas de la fábula. La calidad de los resultados del LLM dependió directamente de la relevancia de las palabras clave de entrada: en las fábulas donde LDA extrajo palabras más específicas (animales, verbos de acción), el LLM generó descripciones más precisas. En las fábulas con palabras clave más genéricas, los enunciados generados fueron más abstractos.

### Reflexión final

La actividad evidenció la importancia del preprocesamiento en pipelines de NLP: la calidad de cada etapa condiciona la siguiente. El enlace más débil fue LDA aplicado a textos cortos, situación que podría resolverse en aplicaciones reales con corpus más extensos o utilizando modelos de extracción de palabras clave más adecuados para documentos cortos (como KeyBERT o YAKE). La integración de Whisper + LDA + GPT-4o demostró ser un pipeline viable para análisis automatizado de contenido de audio en español.

# **Fin de la actividad LDA y LMM: audio-a-texto**